In [0]:
%sql
use catalog aws_s3_catalog;

In [0]:
display(dbutils.fs.ls("s3://sportsdata-analysis/"))

###Read the customer data

In [0]:
df_customers = (spark.read
             .format("csv")              # change if the files are parquet/json
             .option("header", "true")
             .option("inferSchema", "true")
             .load("s3://sportsdata-analysis/customers/"))
display(df_customers)

#create delta table


In [0]:
catalog = "aws_s3_catalog"
bucket  = "s3://sportsdata-analysis"
folders = ["customers", "gross_price", "products"]

for f in folders:
    path = f"{bucket}/{f}/"
    try:
        df = (spark.read
              .format("csv")
              .option("header", "true")
              .option("inferSchema", "true")
              .option("recursiveFileLookup", "true")
              .load(path))

        table = f"{catalog}.bronze.{f}"
        (df.write
           .mode("overwrite")
           .option("overwriteSchema", "true")
           .saveAsTable(table))

        print(f"✅ {f}: {spark.table(table).count()} rows loaded into {table}")

    except Exception as e:
        print(f"❌ {f}: failed – {str(e)[:200]}")

In [0]:
%sql
show tables in aws_s3_catalog.bronze;

In [0]:
for t in ["customers", "gross_price", "products"]:
    print(t, spark.table(f"aws_s3_catalog.bronze.{t}").count())

In [0]:
%sql
select * from aws_s3_catalog.bronze.customers;